In [10]:
import pandas as pd
import numpy as np

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

c:\Users\Josip\Desktop\Data science and AI\Lectures\Week12\Clustering\ds-clustering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
df = pd.read_csv("../data/df_features.csv", parse_dates = ['timestamp'])

In [32]:
df.columns.tolist()

['timestamp',
 'price',
 'load',
 'wind_offshore',
 'wind_onshore',
 'solar',
 'hour',
 'day_of_week',
 'month',
 'temperature',
 'wind_speed',
 'is_weekend',
 'gas_price',
 'coal_price',
 'price_lag_24h',
 'price_lag_168h',
 'price_rolling_24h',
 'price_rolling_168h',
 'co2_price',
 'is_holiday',
 'is_hol_or_week',
 'total_generation',
 'net_export',
 'coal_generation',
 'gas_generation',
 'nuclear_generation',
 'actual_wind_offshore',
 'actual_wind_onshore',
 'actual_solar',
 'actual_load',
 'hour_sin',
 'hour_cos',
 'dow_sin',
 'dow_cos',
 'month_sin',
 'month_cos',
 'gas_price_lag_24h',
 'gas_price_lag_168h',
 'coal_price_lag_24h',
 'coal_price_lag_168h',
 'co2_price_lag_24h',
 'renewable_share',
 'fuel_cost_index',
 'dispatchable_gen',
 'demand_supply_gap',
 'is_peak_hour',
 'wind_x_peak',
 'gas_x_peak',
 'solar_x_demand',
 'renewable_share_x_peak',
 'is_crisis_period',
 'is_high_price_regime',
 'is_negative_price',
 'year',
 'residual_load',
 'load_ramp',
 'renewable_ramp',
 'pri

In [12]:
# Define features and target
EXCLUDE_COLS = [
    'timestamp',
    'price',
    'is_high_price_regime',
    'is_negative_price',
    'actual_wind_offshore',
    'actual_wind_onshore',
    'actual_solar',
    'actual_load',
]

feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]

X = df[feature_cols]
y = df['price']

# Chronological train/test split
split_date = '2025-01-01'

X_train = X[df['timestamp'] < split_date]
X_test  = X[df['timestamp'] >= split_date]
y_train = y[df['timestamp'] < split_date]
y_test  = y[df['timestamp'] >= split_date]

print(f"Features:      {X.shape[1]}")
print(f"Train size:    {len(X_train):,} rows  ({X_train.shape[0] / len(X) * 100:.1f}%)")
print(f"Test size:     {len(X_test):,} rows   ({X_test.shape[0] / len(X) * 100:.1f}%)")
print(f"Train range:   {df[df['timestamp'] < split_date]['timestamp'].min().date()} → {df[df['timestamp'] < split_date]['timestamp'].max().date()}")
print(f"Test range:    {df[df['timestamp'] >= split_date]['timestamp'].min().date()} → {df[df['timestamp'] >= split_date]['timestamp'].max().date()}")

Features:      52
Train size:    52,266 rows  (83.5%)
Test size:     10,294 rows   (16.5%)
Train range:   2019-01-15 → 2024-12-31
Test range:    2025-01-01 → 2026-03-05


In [13]:
def evaluate_model(name, y_true, y_pred):
    # NOTE: using np.sqrt(MSE) instead of mean_squared_error(squared=False)
    # — squared=False was deprecated in sklearn 1.4 and removed in 1.6
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"{name:<25} RMSE: {rmse:7.2f} €/MWh  |  MAE: {mae:7.2f} €/MWh  |  R²: {r2:.4f}")
    return {'model': name, 'rmse': rmse, 'mae': mae, 'r2': r2}

results = []

In [14]:
# --- Baseline 1: DummyRegressor ---
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)
results.append(evaluate_model('Dummy (mean)', y_test, y_pred_dummy))

# --- Baseline 2: Naive (price_lag_24h as prediction) ---
y_pred_naive = X_test['price_lag_24h'].values
results.append(evaluate_model('Naive (lag 24h)', y_test, y_pred_naive))

# --- Baseline 3: Linear Regression ---
# NOTE: RobustScaler only here — tree models below are scale-invariant
lr_pipeline = Pipeline([
    ('scaler', RobustScaler()),
    ('model', LinearRegression())
])
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
results.append(evaluate_model('Linear Regression', y_test, y_pred_lr))

Dummy (mean)              RMSE:   50.56 €/MWh  |  MAE:   34.19 €/MWh  |  R²: -0.0082
Naive (lag 24h)           RMSE:   39.40 €/MWh  |  MAE:   25.34 €/MWh  |  R²: 0.3876
Linear Regression         RMSE:   22.38 €/MWh  |  MAE:   15.51 €/MWh  |  R²: 0.8025


In [15]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

True
NVIDIA GeForce GTX 1650


In [16]:
# NOTE: No RobustScaler — tree models are scale-invariant

# Random Forest — no GPU support in sklearn
# rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
# rf.fit(X_train, y_train)
# results.append(evaluate_model('Random Forest', y_test, rf.predict(X_test)))

# XGBoost — GPU
# xgb_model = XGBRegressor(n_estimators=300, random_state=42, device='cuda', verbosity=0)
# xgb_model.fit(X_train, y_train)
# results.append(evaluate_model('XGBoost', y_test, xgb_model.predict(X_test)))

# LightGBM — GPU
lgbm_model = LGBMRegressor(n_estimators=300, random_state=42, device='gpu', verbose=-1)
lgbm_model.fit(X_train, y_train)
results.append(evaluate_model('LightGBM', y_test, lgbm_model.predict(X_test)))

# CatBoost — GPU
# cat_model = CatBoostRegressor(iterations=300, random_state=42, task_type='GPU', verbose=0)
# cat_model.fit(X_train, y_train)
# results.append(evaluate_model('CatBoost', y_test, cat_model.predict(X_test)))

LightGBM                  RMSE:   18.27 €/MWh  |  MAE:   12.54 €/MWh  |  R²: 0.8684


In [17]:
# Summary table
results_df = pd.DataFrame(results).sort_values('rmse')
print("\n=== MODEL COMPARISON ===")
print(results_df.to_string(index=False))


=== MODEL COMPARISON ===
            model      rmse       mae        r2
         LightGBM 18.266094 12.537094  0.868404
Linear Regression 22.378702 15.507908  0.802476
  Naive (lag 24h) 39.403971 25.344373  0.387607
     Dummy (mean) 50.559294 34.189661 -0.008213


In [18]:
# NOTE: RandomizedSearchCV was an initial exploration step to get a rough sense
# of the hyperparameter space. Superseded by Optuna below — kept for reference only.

In [19]:
# NOTE: GridSearchCV was a focused follow-up around promising values from
# RandomizedSearchCV. Superseded by Optuna below — kept for reference only.

In [21]:
tscv = TimeSeriesSplit(n_splits=5, gap=168)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 200),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'device': 'gpu',
        'random_state': 42,
        'verbose': -1,
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
    }

    model = LGBMRegressor(**params)

    scores = cross_val_score(
        model, X_train, y_train,
        cv=tscv,
        scoring='neg_root_mean_squared_error'
    )
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nBest RMSE (CV): {-study.best_value:.2f} €/MWh")
print(f"Best params:    {study.best_params}")

Best trial: 49. Best value: -45.8333: 100%|██████████| 50/50 [44:54<00:00, 53.88s/it] 


Best RMSE (CV): 45.83 €/MWh
Best params:    {'n_estimators': 301, 'max_depth': 12, 'learning_rate': 0.06438493832581466, 'num_leaves': 73, 'subsample': 0.998198716248151, 'colsample_bytree': 0.8180558276228386, 'reg_alpha': 3.139097481238134e-05, 'reg_lambda': 0.004030329597131774, 'min_child_samples': 94}


In [22]:
tscv_check = TimeSeriesSplit(n_splits=5, gap=168)
for fold, (train_idx, val_idx) in enumerate(tscv_check.split(X_train)):
    val_dates = df[df['timestamp'] < '2025-01-01']['timestamp'].iloc[val_idx]
    print(f"Fold {fold+1}: val {val_dates.min().date()} → {val_dates.max().date()}  ({len(val_idx):,} rows)")

Fold 1: val 2020-01-13 → 2021-01-09  (8,711 rows)
Fold 2: val 2021-01-10 → 2022-01-07  (8,711 rows)
Fold 3: val 2022-01-08 → 2023-01-05  (8,711 rows)
Fold 4: val 2023-01-06 → 2024-01-03  (8,711 rows)
Fold 5: val 2024-01-04 → 2024-12-31  (8,711 rows)


In [23]:
# visual for code block above
for train_idx, val_idx in tscv_check.split(X_train):
    print(train_idx[:5], "...", train_idx[-5:])
    print(val_idx[:5], "...", val_idx[-5:])
    print()

[0 1 2 3 4] ... [8538 8539 8540 8541 8542]
[8711 8712 8713 8714 8715] ... [17417 17418 17419 17420 17421]

[0 1 2 3 4] ... [17249 17250 17251 17252 17253]
[17422 17423 17424 17425 17426] ... [26128 26129 26130 26131 26132]

[0 1 2 3 4] ... [25960 25961 25962 25963 25964]
[26133 26134 26135 26136 26137] ... [34839 34840 34841 34842 34843]

[0 1 2 3 4] ... [34671 34672 34673 34674 34675]
[34844 34845 34846 34847 34848] ... [43550 43551 43552 43553 43554]

[0 1 2 3 4] ... [43382 43383 43384 43385 43386]
[43555 43556 43557 43558 43559] ... [52261 52262 52263 52264 52265]



In [24]:
# Use only last fold for CV — closest to test period
tscv_last = TimeSeriesSplit(n_splits=5, gap=168)

splits = list(tscv_last.split(X_train))
last_train_idx, last_val_idx = splits[-1]

X_tr = X_train.iloc[last_train_idx]
y_tr = y_train.iloc[last_train_idx]
X_val = X_train.iloc[last_val_idx]
y_val = y_train.iloc[last_val_idx]

print(f"Val period: {df[df['timestamp'] < '2025-01-01']['timestamp'].iloc[last_val_idx].min().date()} → {df[df['timestamp'] < '2025-01-01']['timestamp'].iloc[last_val_idx].max().date()}")

# Quick sanity check with default LightGBM on last fold
lgbm_check = LGBMRegressor(device='gpu', random_state=42, verbose=-1)
lgbm_check.fit(X_tr, y_tr)
y_pred_val = lgbm_check.predict(X_val)

rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
print(f"RMSE on last fold (2024): {rmse_val:.2f} €/MWh")

Val period: 2024-01-04 → 2024-12-31
RMSE on last fold (2024): 23.21 €/MWh


In [ ]:
# NOTE: This was an alternative Optuna study using only the last fold for faster
# iteration. The full 5-fold study above is the canonical run — study.best_params
# used in the final model comes from there.

In [30]:
# Fit final model with best params from Optuna on entire train set
# NOTE: device='gpu' must be set explicitly — it is not part of study.best_params
# since it was a fixed param in the Optuna objective, not a tuned one
final_lgbm = LGBMRegressor(
    **study.best_params,
    device='gpu',
    random_state=42,
    verbose=-1
)
final_lgbm.fit(X_train, y_train)

y_pred_tuned = final_lgbm.predict(X_test)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
mae_tuned  = mean_absolute_error(y_test, y_pred_tuned)
r2_tuned   = r2_score(y_test, y_pred_tuned)

# Compare against default LightGBM from the results table (no hardcoded values)
default_lgbm = next(r for r in results if r['model'] == 'LightGBM')
print(f"Default LightGBM:  RMSE: {default_lgbm['rmse']:.2f} €/MWh  |  MAE: {default_lgbm['mae']:.2f} €/MWh  |  R²: {default_lgbm['r2']:.4f}")
print(f"Tuned LightGBM:    RMSE: {rmse_tuned:.2f} €/MWh  |  MAE: {mae_tuned:.2f} €/MWh  |  R²: {r2_tuned:.4f}")

Default LightGBM:  RMSE: 18.27 €/MWh  |  MAE: 12.54 €/MWh  |  R²: 0.8684
Tuned LightGBM:    RMSE: 18.53 €/MWh  |  MAE: 12.51 €/MWh  |  R²: 0.8646


In [ ]:
# NOTE: Cross-border flow validation (verifying net_export against raw ENTSO-E CSV)
# was done here during data prep debugging. Belongs in data preparation, not modeling.